# Theory

The test set is the final, untouched holdout used exactly once — to report how well the finished model is expected to perform on genuinely new cars it has never encountered in any form, not during training and not during hyperparameter tuning.

This is why the workflow strictly separates:

Train → fit the model's parameters.
Validation → choose between models/hyperparameters.
Test → touched once, at the very end, to report the number that goes in the final report.

Applying It to This Dataset
**bold text**
For used_cars_dataset.csv, imagine our full workflow was:

Split off 20% of listings as X_test / y_test right at the start.
Use the remaining 80% for training and cross-validated hyperparameter tuning

Pick the best model configuration (say, RandomForestRegressor(n_estimators=300)).
Only now do we touch X_test — to report the number that tells us "if this model were deployed to price real used-car listings tomorrow, this is roughly the error we'd expect."

# What "Success" Looks Like Here

Suppose the final test evaluation gives a Mean Absolute Error (MAE) of Rs. 180,000 on Price. That means: on average, the model's price prediction is off by about Rs. 180,000 per car. Whether that's "good enough" depends entirely on context — Rs. 180,000 error is trivial for a car worth Rs. 13,000,000 but serious for one worth Rs. 300,000. This is why looking at metrics relative to price ranges (or using percentage-based metrics like MAPE) often tells a fuller story than a single flat error number.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

df = pd.read_csv("used_cars_dataset.csv").drop(columns=["Car_ID"])
X = df.drop(columns=["Price"])
y = df["Price"]

# Step 1: split off the test set FIRST, before any tuning happens
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_features = ["Brand", "Model", "Fuel_Type", "Transmission", "Owner_Type", "Color", "City"]
preprocessor = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
    remainder="passthrough"
)

# Step 2 & 3: tuning happens on X_trainval only (cross-validation, see Lecture 58) — omitted here for brevity
# Assume RandomForestRegressor(n_estimators=300) was selected as the winner

final_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(n_estimators=300, random_state=42))
])

# Fit the final model on ALL of train+validation data (now that tuning is done)
final_pipeline.fit(X_trainval, y_trainval)

# Step 4: evaluate ONCE on the untouched test set
test_preds = final_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, test_preds)
mape = mean_absolute_percentage_error(y_test, test_preds)
r2 = r2_score(y_test, test_preds)

print(f"Final Test MAE:  Rs. {mae:,.0f}")
print(f"Final Test MAPE: {mape:.2%}")
print(f"Final Test R²:   {r2:.4f}")

**Visual Check: Predicted vs. Actual Price**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))
plt.scatter(y_test, test_preds, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Test Set: Predicted vs. Actual Price")
plt.show()

*A tight cluster around the diagonal red line indicates accurate predictions; systematic deviation (e.g., the model consistently under-predicting high-end cars) reveals where the model still struggles.*